# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
#Write your code below.
from dotenv import load_dotenv

load_dotenv()
import os
print(os.getenv("PRICE_DATA"))


../../05_src/data/prices/


In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")

parquet_files = glob(
    os.path.join(PRICE_DATA, "**/*.parquet"),
    recursive=True
)

print("PRICE_DATA:", PRICE_DATA)
print("Number of parquet files:", len(parquet_files))
print("First few files:", parquet_files[:3])



PRICE_DATA: ../../05_src/data/prices/
Number of parquet files: 2010
First few files: ['../../05_src/data/prices/BKTI/BKTI_2012/part.0.parquet', '../../05_src/data/prices/BKTI/BKTI_2012/part.1.parquet', '../../05_src/data/prices/BKTI/BKTI_2015/part.0.parquet']


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
# Write your code below.
# Load the stock price data from Parquet files
dd_px = dd.read_parquet(parquet_files)

print(dd_px.columns)
#test print to see if the files are properly read and table is properly loaded 
#print(dd_px.head(4))
dd_px.head()

# Identify the adjusted closing price column, which may have different names.
if "Adj_Close" in dd_px.columns:
    adj_col = "Adj_Close"
elif "Adj Close" in dd_px.columns:
    adj_col = "Adj Close"
else:
    raise KeyError("Could not find an adjusted-close column.")

print("Using adjusted-close column:", adj_col)

# Set ticker as the index to organize the data by individual stock.
dd_px = dd_px.set_index("ticker")
##print to see if table has properly been updated/using the index
#print(dd_px.head(4))

# Sort each stock's observations by date and calculate one-period price lags.
import pandas as pd

##sort each ticker by Date first, then shift within that ticker.
def add_lags(ticker_df):
    ticker_df = ticker_df.sort_values("Date", ascending=True)
    return ticker_df.assign(
        Close_lag_1=ticker_df["Close"].shift(1),
        Adj_Close_lag_1=ticker_df[adj_col].shift(1)
    )

meta = dd_px._meta.assign(
    Close_lag_1=pd.Series(dtype="float64"),
    Adj_Close_lag_1=pd.Series(dtype="float64")
)

dd_shift = (
    dd_px
    .groupby("ticker", group_keys=False)
    .apply(add_lags, meta=meta)
)
#groupby('ticker') separates one stock from another.
#sort_values('Date') puts each ticker in chronological order.
#shift(1) moves the previous day's value down one row.
#meta tells Dask what the resulting columns/types will look like before the lazy computation runs

# Add two new feature columns to the existing Dask DataFrame.
# assign() returns a new DataFrame without modifying dd_shift.
dd_feat = dd_shift.assign(

    # Calculate daily returns using today's Close and the previous day's Close.
    # lambda x receives the DataFrame being processed by assign().
    returns=lambda x: (x["Close"] / x["Close_lag_1"]) - 1,

    # Calculate the daily price range by subtracting Low from High.
    hi_lo_range=lambda x: x["High"] - x["Low"]
)

# Preview the resulting DataFrame with the newly created features.
print(dd_feat.head())
print(dd_feat.columns.tolist())

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker', 'Year'],
      dtype='str')
Using adjusted-close column: Adj Close
             Date       Open       High        Low      Close  Adj Close  \
ticker                                                                     
A      1999-11-18  32.546494  35.765381  28.612303  31.473534  27.068665   
A      1999-11-19  30.713520  30.758226  28.478184  28.880543  24.838577   
A      1999-11-22  29.551144  31.473534  28.657009  31.473534  27.068665   
A      1999-11-23  30.400572  31.205294  28.612303  28.612303  24.607880   
A      1999-11-24  28.701717  29.998211  28.612303  29.372318  25.261524   

            Volume source  Year  Close_lag_1  Adj_Close_lag_1   returns  \
ticker                                                                    
A       62546300.0  A.csv  1999          NaN              NaN       NaN   
A       15234100.0  A.csv  1999    31.473534        27.068665 -0.082386   
A  

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [5]:
# Write your code below.
# compute() actually executes the Dask task dataframe and materializes the data in memory as pandas. 
# reset_index() makes ticker a regular column again so the next steps are easier to read and easier to specify.
df_feat = dd_feat.compute().reset_index()

# sort by ticker symbol and date: 
df_feat = df_feat.sort_values(["ticker", "Date"])

#Calculate the 10-day moving average separately for each ticker
df_feat["returns_ma_10"] = (
    df_feat
    .groupby("ticker")["returns"]
    .transform(lambda s: s.rolling(10).mean())
)

# Inspect the result
df_feat[
    ["ticker", "Date", "returns", "returns_ma_10"]
].head(15)


,ticker,Date,returns,returns_ma_10
0,A,1999-11-18,NaN,NaN
1,A,1999-11-19,-0.082386,NaN
2,A,1999-11-22,0.089783,NaN
3,A,1999-11-23,-0.090909,NaN
4,A,1999-11-24,0.026563,NaN
5,A,1999-11-26,0.003044,NaN
6,A,1999-11-29,0.022762,NaN
7,A,1999-11-30,0.001484,NaN
8,A,1999-12-01,0.017778,NaN
9,A,1999-12-02,0.027656,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

No, converting to pandas was not required because Dask can also calculate rolling averages. However, pandas is simpler when the dataset is small enough to fit in computer memory. Dask is more useful for larger datasets because it can process data in smaller parts. Rolling calculations can be more complicated in Dask when they need data from different parts.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.